# Detecting Water-Temperature and Streamflow Anomalies for Fish Habitat Monitoring

This demo notebook downloads USGS water-temperature and streamflow data, creates a daily dataset, saves a CSV that can be used in **Orange**, and also demonstrates a simple AI anomaly-detection model in Python.

Suggested site: **USGS 01186000 — West Branch Farmington River at Riverton, CT**

Variables:
- Water temperature: USGS parameter `00010`, degrees Celsius
- Discharge / streamflow: USGS parameter `00060`, cubic feet per second


In [ ]:
# Install requirements if needed.
# In Google Colab, pandas, numpy, matplotlib, and scikit-learn are usually already available.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path


## 1. Set dataset parameters

You can change the site or date range, but the default is designed to keep the dataset small and relevant for a summer fish-habitat monitoring example.


In [ ]:
SITE = "01186000"
START_DATE = "2024-06-01"
END_DATE = "2024-09-30"
PARAMS = "00010,00060"  # 00010 = water temperature C; 00060 = discharge cfs

url = (
    "https://waterservices.usgs.gov/nwis/iv/"
    f"?format=rdb&sites={SITE}&parameterCd={PARAMS}"
    f"&startDT={START_DATE}&endDT={END_DATE}&siteStatus=all"
)

print(url)


## 2. Download data from USGS

The data are downloaded in USGS RDB tab-delimited format. The code below removes the USGS type-code row and finds the value columns for water temperature and discharge.


In [ ]:
raw = pd.read_csv(url, sep="\t", comment="#", dtype=str)

# USGS RDB files include a second row with column-width/type codes, often beginning with "5s".
if "agency_cd" in raw.columns:
    raw = raw[raw["agency_cd"] != "5s"].copy()
else:
    raw = raw[raw.iloc[:, 0] != "5s"].copy()

print("Columns:")
print(raw.columns.tolist())
print("\nFirst rows:")
display(raw.head())


In [ ]:
def find_value_column(columns, parameter_code):
    """Find the numeric value column for a USGS parameter code."""
    candidates = [
        c for c in columns
        if parameter_code in c and not c.lower().endswith("_cd")
    ]
    if not candidates:
        raise ValueError(f"Could not find value column for parameter {parameter_code}. Columns: {list(columns)}")
    return candidates[0]

temp_col = find_value_column(raw.columns, "00010")
flow_col = find_value_column(raw.columns, "00060")

print("Temperature column:", temp_col)
print("Flow column:", flow_col)


## 3. Clean and summarize to daily values

Orange works best with a simple table. Here we convert the 15-minute or hourly observations into daily features.


In [ ]:
df = pd.DataFrame({
    "datetime": pd.to_datetime(raw["datetime"], utc=True, errors="coerce").dt.tz_convert("America/New_York"),
    "water_temp_C": pd.to_numeric(raw[temp_col], errors="coerce"),
    "discharge_cfs": pd.to_numeric(raw[flow_col], errors="coerce")
}).dropna()

df = df.set_index("datetime").sort_index()

print(df.head())
print(df.tail())


In [ ]:
daily = df.resample("D").agg(
    temp_mean_C=("water_temp_C", "mean"),
    temp_max_C=("water_temp_C", "max"),
    temp_min_C=("water_temp_C", "min"),
    flow_mean_cfs=("discharge_cfs", "mean"),
    flow_min_cfs=("discharge_cfs", "min"),
    flow_max_cfs=("discharge_cfs", "max"),
    n_obs=("water_temp_C", "count")
)

daily["temp_range_C"] = daily["temp_max_C"] - daily["temp_min_C"]
daily["flow_log_mean"] = np.log1p(daily["flow_mean_cfs"])
daily["temp_change_1d"] = daily["temp_mean_C"].diff()
daily["flow_change_1d"] = daily["flow_mean_cfs"].diff()
daily["day_of_year"] = daily.index.dayofyear
daily["month"] = daily.index.month

daily = daily.dropna()

display(daily.head())
display(daily.describe())


## 4. Save an Orange-ready CSV

Use this CSV in Orange. Load it with **File** or **CSV File Import**. In Orange, treat `date` as a meta variable or ignore it for modeling.


In [ ]:
daily_out = daily.reset_index()
daily_out["date"] = daily_out["datetime"].dt.date.astype(str)
daily_out = daily_out.drop(columns=["datetime"])

orange_cols = [
    "date",
    "temp_mean_C",
    "temp_max_C",
    "temp_min_C",
    "temp_range_C",
    "flow_mean_cfs",
    "flow_min_cfs",
    "flow_max_cfs",
    "flow_log_mean",
    "temp_change_1d",
    "flow_change_1d",
    "day_of_year",
    "month",
    "n_obs"
]

orange_csv = "usgs_water_fish_habitat_orange_ready.csv"
daily_out[orange_cols].to_csv(orange_csv, index=False)

print("Saved:", orange_csv)
display(daily_out[orange_cols].head())


### Optional: Download the CSV from Colab

Run this cell in Google Colab if you want to download the CSV to your computer and then open it in Orange.


In [ ]:
try:
    from google.colab import files
    files.download(orange_csv)
except Exception:
    print("This download helper works in Google Colab. If running locally, find the CSV in the current folder.")


## 5. Simple AI model in Colab: Isolation Forest

This is the same kind of model students can use in Orange through the **Outliers** widget. Isolation Forest is unsupervised: it learns common patterns in the feature table and flags unusual rows.


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

model_features = [
    "temp_mean_C",
    "temp_max_C",
    "temp_range_C",
    "flow_mean_cfs",
    "flow_log_mean",
    "temp_change_1d",
    "flow_change_1d",
    "day_of_year"
]

X = daily_out[model_features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = IsolationForest(
    n_estimators=200,
    contamination=0.05,  # roughly 5% of days are flagged as unusual
    random_state=42
)

daily_out["ai_outlier_code"] = model.fit_predict(X_scaled)
daily_out["ai_anomaly_label"] = np.where(daily_out["ai_outlier_code"] == -1, "anomaly", "normal")
daily_out["anomaly_score"] = -model.decision_function(X_scaled)

display(daily_out["ai_anomaly_label"].value_counts())
display(daily_out.sort_values("anomaly_score", ascending=False).head(10)[
    ["date", "temp_max_C", "flow_mean_cfs", "temp_change_1d", "flow_change_1d", "ai_anomaly_label", "anomaly_score"]
])


## 6. Visualize AI-flagged anomalies

The first plot shows daily maximum water temperature and highlights days flagged by the AI model. The second plot shows temperature vs. streamflow, which can help students interpret whether anomalies are linked to warm water, low flow, high flow, or combinations of conditions.


In [ ]:
anoms = daily_out[daily_out["ai_anomaly_label"] == "anomaly"]

plt.figure(figsize=(12, 5))
plt.plot(pd.to_datetime(daily_out["date"]), daily_out["temp_max_C"], label="Daily max water temperature")
plt.scatter(pd.to_datetime(anoms["date"]), anoms["temp_max_C"], marker="x", s=80, label="AI-flagged anomaly")
plt.xlabel("Date")
plt.ylabel("Water temperature (°C)")
plt.title("AI-Flagged Anomalies in Daily Maximum Water Temperature")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
normal = daily_out[daily_out["ai_anomaly_label"] == "normal"]

plt.scatter(normal["flow_mean_cfs"], normal["temp_max_C"], label="Normal")
plt.scatter(anoms["flow_mean_cfs"], anoms["temp_max_C"], marker="x", s=80, label="AI-flagged anomaly")
plt.xlabel("Mean daily streamflow (cfs)")
plt.ylabel("Maximum daily water temperature (°C)")
plt.title("Temperature vs. Streamflow")
plt.legend()
plt.show()


## 7. Save AI-flagged results

This file can be submitted as evidence of the model output or compared with the Orange output.


In [ ]:
ai_csv = "usgs_water_fish_habitat_with_ai_flags.csv"
daily_out.to_csv(ai_csv, index=False)
print("Saved:", ai_csv)

try:
    from google.colab import files
    files.download(ai_csv)
except Exception:
    print("This download helper works in Google Colab. If running locally, find the CSV in the current folder.")


## 8. Interpretation prompt

Write 250–400 words responding to the following:

1. What kinds of days did the AI model flag as anomalies?
2. Were the anomalies mostly related to temperature, streamflow, sudden changes, or combinations of conditions?
3. Why could these conditions matter for fish habitat?
4. What are the limitations of using this simple model?

Remember: the model flags unusual days. It does not prove that fish were harmed. A real assessment would require ecological thresholds, species information, expert review, and additional field data.
